# SmartTrip AI — 02 · Recommandation des activités

**Objectif :** charger `all_activites`, auditer les valeurs manquantes et les doublons, nettoyer sans supprimer inutilement les activités, préparer les catégories et recommander des activités selon le profil du premier notebook.

Exécuter les cellules **dans l’ordre** sur Kaggle. Ajouter `all_activites.csv` dans **Add Input**. Le profil JSON est facultatif : un profil de test est prévu.

**Attention :** `fee` ne représente pas un prix chiffré, les horaires et descriptions sont souvent absents. Le classement est fondé sur les catégories et la complétude, **pas** sur une note de qualité ni une garantie d'ouverture.

## 1 — Charger le dataset

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

fichiers = list(
    Path("/kaggle/input").rglob("all_activites_clean_v3.csv")
)

if not fichiers:
    raise FileNotFoundError(
        "Ajouter all_activites_clean_v3.csv dans les inputs"
    )

activites_df = pd.read_csv(fichiers[0])

print("Dataset chargé :", fichiers[0])
print("Dimensions :", activites_df.shape)

display(activites_df.head())

/tmp/ipykernel_50/2567779154.py:14: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  activites_df = pd.read_csv(fichiers[0])


Dataset chargé : /kaggle/input/datasets/areejsallami/all-activites-clean-v3/all_activites_clean_v3.csv
Dimensions : (311600, 24)


,country,destination,osm_id,name,category,latitude,longitude,fee,opening_hours,website,...,category_norm,completude,preference_smarttrip,country_original,category_en,ecriture_nom,langue_source,methode_langue,name_en,name_fr
0,United Arab Emirates,Abu Dhabi,relation/7232861,قصر الوطن,castle,24.463287,54.305442,yes,11:00-19:00,https://www.qasralwatan.ae/,...,castle,3,culture,United Arab Emirates,Attractions,Arabic,ar,geographie,Al-Watan Palace,NaN
1,United Arab Emirates,Abu Dhabi,node/8327303399,TEN 11 Coffee Boutique,cafe,24.476411,54.321555,NaN,Mo-Su 10:00-22:00,https://www.linkedin.com/company/ten11-coffee-...,...,cafe,3,gastronomy,United Arab Emirates,Food & Drink,Latin,ar,geographie,NaN,NaN
2,United Arab Emirates,Abu Dhabi,node/8327349672,TEN 11 Beach,cafe,24.538983,54.408144,NaN,Mo-Su 10:00-22:00,https://www.ten11cafe.ae/,...,cafe,3,gastronomy,United Arab Emirates,Food & Drink,Latin,ar,geographie,NaN,NaN
3,Turkey,Antalya,way/1117210531,"Bir Bakışta Türkiye, Açık Hava Müzesi",museum,36.912009,30.676158,no,08:30-20:30,https://kepezdokuma.com/,...,museum,3,culture,Turkey,Museums,Latin,tr,geographie,NaN,NaN
4,Turkey,Antalya,node/3574829988,Queen Sahne Antalya,restaurant,36.884318,30.686107,NaN,24/7,https://instagram.com/queensahneantalya,...,restaurant,3,gastronomy,Turkey,Food & Drink,Latin,tr,geographie,NaN,NaN


In [23]:
for col in ["name_en", "name_fr"]:
    activites_df[col] = (
        activites_df[col]
        .astype("string")
        .str.strip()
        .replace(["", "None", "nan"], pd.NA)
    )

activites_df["name_display"] = (
    activites_df["name_en"]
    .combine_first(activites_df["name_fr"])
    .combine_first(activites_df["name"])
)

display(
    activites_df[
        ["name", "name_en", "name_fr", "name_display"]
    ].head(20)
)

,name,name_en,name_fr,name_display
0,قصر الوطن,Al-Watan Palace,<NA>,Al-Watan Palace
1,TEN 11 Coffee Boutique,<NA>,<NA>,TEN 11 Coffee Boutique
2,TEN 11 Beach,<NA>,<NA>,TEN 11 Beach
3,"Bir Bakışta Türkiye, Açık Hava Müzesi",<NA>,<NA>,"Bir Bakışta Türkiye, Açık Hava Müzesi"
4,Queen Sahne Antalya,<NA>,<NA>,Queen Sahne Antalya
5,Forellenhof Göynük Fish & Steak House,<NA>,<NA>,Forellenhof Göynük Fish & Steak House
6,Charms Coffee,<NA>,<NA>,Charms Coffee
7,NesBu Cafe Restaurant,<NA>,<NA>,NesBu Cafe Restaurant
8,Leman's Deli,Leman's Deli,<NA>,Leman's Deli
9,พันธุ์ทิพย์พลาซ่า,Phenix,<NA>,Phenix


## 2 — Audit des valeurs manquantes et des doublons (sans modifier les données)

In [24]:
colonnes_requises = ['country','destination','osm_id','name','category','latitude','longitude']
absentes = [c for c in colonnes_requises if c not in activites_df.columns]
if absentes:
    raise ValueError(f'Colonnes obligatoires absentes : {absentes}')

manquants = pd.DataFrame({
    'valeurs_manquantes': activites_df.isna().sum(),
    'pourcentage': (activites_df.isna().mean()*100).round(2)
}).sort_values('valeurs_manquantes', ascending=False)
display(manquants)

def normaliser(serie):
    return (serie.astype('string').str.strip().str.lower()
            .str.replace(r'\s+', ' ', regex=True))

audit = activites_df.copy()
for col in ['country','destination','name']:
    audit[col+'_norm'] = normaliser(audit[col])
cles = ['country_norm','destination_norm','name_norm']
valides = audit.dropna(subset=cles).copy()
valides = valides[valides['name_norm'].ne('')]
print('Lignes exactement identiques :', activites_df.duplicated().sum())
print('OSM ID répétés (à vérifier) :', activites_df.duplicated('osm_id').sum())
print('Lignes avec nom répété dans même pays et ville :',
      valides.duplicated(cles, keep=False).sum())
print('Occurrences supplémentaires de ces noms :',
      valides.duplicated(cles).sum())

doublons_noms = valides[valides.duplicated(cles, keep=False)]
display(doublons_noms[['country','destination','name','category','osm_id','latitude','longitude']].head(30))

,valeurs_manquantes,pourcentage
fee,309403,99.29
description,306790,98.46
name_fr,306700,98.43
name_en,276525,88.74
website,256350,82.27
opening_hours,253493,81.35
preference_smarttrip,10664,3.42
destination,0,0.00
country,0,0.00
latitude,0,0.00


Lignes exactement identiques : 0
OSM ID répétés (à vérifier) : 0
Lignes avec nom répété dans même pays et ville : 49784
Occurrences supplémentaires de ces noms : 35818


,country,destination,name,category,osm_id,latitude,longitude
1,United Arab Emirates,Abu Dhabi,TEN 11 Coffee Boutique,cafe,node/8327303399,24.476411,54.321555
75,Germany,Berlin,Syrtaki,restaurant,way/258342170,52.592645,13.454810
80,Germany,Berlin,The Coffee Shop,cafe,node/305259644,52.515259,13.398318
102,Germany,Berlin,La Luna,restaurant,node/610198284,52.414258,13.400940
104,Germany,Berlin,Coffee Star,cafe,node/638214074,52.535629,13.421401
108,Germany,Berlin,Zia Maria,restaurant,node/745154923,52.505252,13.469174
116,Germany,Berlin,Flying Roasters,cafe,node/832840930,52.491514,13.396281
117,Germany,Berlin,Flying Roasters,cafe,node/5934239970,52.543818,13.377077
136,Germany,Berlin,Aki Tatsu,restaurant,node/1915596857,52.497237,13.354927
144,Germany,Berlin,King King,restaurant,node/2663357561,52.525170,13.410308


## 3 — Nettoyer les valeurs manquantes

- On écarte uniquement les lignes sans **pays, destination, nom ou catégorie**, ou sans coordonnées valides pour les itinéraires.
- On conserve `fee`, `opening_hours`, `website` et `description` même s'ils sont manquants.
- On conserve le nom d'origine pour l'affichage, et on crée des colonnes normalisées pour les comparaisons.

In [25]:
data = activites_df.copy()
initial = len(data)
for col in ['country','destination','name','category']:
    data[col+'_norm'] = normaliser(data[col])

essentielles = ['country_norm','destination_norm','name_norm','category_norm']
data = data.dropna(subset=essentielles).copy()
for col in essentielles:
    data = data[data[col].ne('')].copy()

for col in ['latitude','longitude']:
    data[col] = pd.to_numeric(data[col], errors='coerce')
data = data[
    data['latitude'].between(-90,90) & data['longitude'].between(-180,180)
].copy()

data['completude'] = (
    data['website'].notna().astype(int)
    + data['opening_hours'].notna().astype(int)
    + data['description'].notna().astype(int)
)
print('Lignes écartées (informations essentielles / GPS invalides) :', initial-len(data))
print('Lignes restantes :',len(data))

Lignes écartées (informations essentielles / GPS invalides) : 0
Lignes restantes : 311600


## 4 — Corriger les doublons probables avec le GPS

Deux lieux avec le même nom dans la même ville **ne sont pas nécessairement des doublons**. Ici, on fusionne seulement les fiches de **même pays, même ville et même nom**, situées à **100 m maximum** de la fiche conservée. Une comparaison directe des distances évite les fusions en chaîne de DBSCAN.

C'est une **heuristique**, pas une preuve : deux cafés homonymes à moins de 100 m peuvent être distincts. Les lignes écartées sont sauvegardées pour contrôle.

In [26]:
from sklearn.metrics.pairwise import haversine_distances

RAYON_KM = 0.1  # 100 mètres, paramètre modifiable
TERRE_KM = 6371.0088

# Préférer les fiches les plus documentées ; garder les données d'origine
ordre = data.sort_values('completude', ascending=False, kind='stable')
cles = ['country_norm','destination_norm','name_norm']
conserver = []
ecarter = []

for _, groupe in ordre.groupby(cles, sort=False, dropna=False):
    # Les fiches sont déjà classées par complétude décroissante
    retenues_coords = []
    for idx, ligne in groupe.iterrows():
        coord = np.radians([ligne['latitude'], ligne['longitude']])
        if retenues_coords:
            distances = haversine_distances(
                coord.reshape(1,2), np.array(retenues_coords)
            )[0] * TERRE_KM
            if np.any(distances <= RAYON_KM):
                ecarter.append(idx)
                continue
        conserver.append(idx)
        retenues_coords.append(coord)

activites_clean = ordre.loc[conserver].copy().reset_index(drop=True)
doublons_a_verifier = ordre.loc[ecarter].copy().reset_index(drop=True)
print('Avant dédoublonnage :', len(data))
print('Après dédoublonnage :', len(activites_clean))
print('Fiches potentiellement redondantes écartées :', len(doublons_a_verifier))
display(doublons_a_verifier[['country','destination','name','category','osm_id','latitude','longitude']].head(20))

Avant dédoublonnage : 311600
Après dédoublonnage : 311600
Fiches potentiellement redondantes écartées : 0


,country,destination,name,category,osm_id,latitude,longitude


In [27]:
# ETAPE 4 BIS — Corriger les doublons OSM restants

print("Avant :", len(activites_clean))

# Priorité aux pays connus
activites_clean["priorite_pays"] = (
    activites_clean["country"]
    .fillna("Unknown")
    .ne("Unknown")
    .astype(int)
)

# Trier par pays connu puis par complétude
activites_clean = activites_clean.sort_values(
    ["priorite_pays", "completude"],
    ascending=[False, False],
    kind="stable"
)

# Écarter uniquement les répétitions
# avec le même identifiant ET les mêmes coordonnées
activites_clean = activites_clean.drop_duplicates(
    subset=["osm_id", "latitude", "longitude"],
    keep="first"
)

# Supprimer la colonne temporaire
activites_clean = (
    activites_clean
    .drop(columns=["priorite_pays"])
    .reset_index(drop=True)
)

print("Après :", len(activites_clean))
print(
    "Doublons OSM restants :",
    activites_clean["osm_id"].duplicated().sum()
)

Avant : 311600
Après : 311600
Doublons OSM restants : 0


In [28]:
activites_clean.to_csv(
    "/kaggle/working/all_activites_clean_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Activités :", len(activites_clean))

Activités : 311600


## 5 — Préparer les catégories pour SmartTrip AI

In [29]:
categories_smarttrip = {
    'culture': ['museum','monument','castle','gallery','artwork',
                'archaeological_site','heritage','ruins','place_of_worship',
                'church','mosque','monastery','memorial'],
    'nature': ['park','garden','beach','peak','hill','waterfall','viewpoint','picnic_site'],
    'adventure': ['hiking','scuba_diving','downhill','dive_centre','sports_centre','theme_park'],
    'gastronomy': ['restaurant','cafe','marketplace'],
    'nightlife': ['bar','dance','amusement_arcade']
}

mapping_inverse = {
    categorie: preference
    for preference, categories in categories_smarttrip.items()
    for categorie in categories
}
activites_clean['preference_smarttrip'] = activites_clean['category_norm'].map(mapping_inverse)
print('Répartition des préférences (NaN = catégorie non classée) :')
display(activites_clean['preference_smarttrip'].value_counts(dropna=False))
print('Catégories non classées les plus fréquentes :')
display(activites_clean.loc[activites_clean['preference_smarttrip'].isna(), 'category_norm'].value_counts().head(20))

Répartition des préférences (NaN = catégorie non classée) :


preference_smarttrip
gastronomy    211870
nature         42450
culture        28958
nightlife      16893
NaN            10664
adventure        765
Name: count, dtype: int64

Catégories non classées les plus fréquentes :


category_norm
attraction         7201
mall               2988
hotel               145
yes                 112
guest_house          33
building             21
beach_resort         21
checkpoint           21
wayside_shrine       13
chapel                7
hostel                7
marina                7
camp_site             6
fitness_centre        6
indoor_play           5
outdoor_seating       5
zoo                   4
fort                  3
altar                 3
parking               3
Name: count, dtype: Int64

In [30]:
activites_clean.to_csv(
    "/kaggle/working/all_activites_clean_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Activités :", len(activites_clean))

Activités : 311600


In [31]:

# ETAPE 5 BIS : Classification jour / nuit

categories_jour = [
    "museum", "park", "garden", "beach",
    "castle", "monument", "hiking",
    "viewpoint", "waterfall", "peak",
    "gallery", "archaeological_site"
]

categories_soir = [
    "restaurant", "bar", "cafe",
    "dance", "amusement_arcade"
]

def classifier_moment(category):
    if category in categories_jour:
        return "jour"
    elif category in categories_soir:
        return "soir_possible"
    else:
        return "a_verifier"

activites_clean["moment"] = (
    activites_clean["category_norm"]
    .apply(classifier_moment)
)

print(activites_clean["moment"].value_counts())

display(
    activites_clean[
        ["name", "category", "moment", "destination"]
    ].head(15)
)


moment
soir_possible    228755
jour              49146
a_verifier        33699
Name: count, dtype: int64


,name,category,moment,destination
0,قصر الوطن,castle,jour,Abu Dhabi
1,TEN 11 Coffee Boutique,cafe,soir_possible,Abu Dhabi
2,TEN 11 Beach,cafe,soir_possible,Abu Dhabi
3,"Bir Bakışta Türkiye, Açık Hava Müzesi",museum,jour,Antalya
4,Queen Sahne Antalya,restaurant,soir_possible,Antalya
5,Forellenhof Göynük Fish & Steak House,restaurant,soir_possible,Antalya
6,Charms Coffee,cafe,soir_possible,Antalya
7,NesBu Cafe Restaurant,cafe,soir_possible,Antalya
8,Leman's Deli,restaurant,soir_possible,Antalya
9,พันธุ์ทิพย์พลาซ่า,mall,a_verifier,Bangkok


## 6 — Charger le profil JSON (ou tester sans JSON)

In [32]:
import json
profils = list(Path('/kaggle/input').rglob('profil_utilisateur (1).json'))
profils += list(Path('/kaggle/working').glob('profil_utilisateur.json'))

if profils:
    with open(profils[0], encoding='utf-8') as f:
        profil = json.load(f)
    print('Profil chargé :', profils[0])
else:
    profil = {
        'destination_choisie': 'Djerba',
        'interets': ['culture','nature'],
        'jours': 5, 'budget_total': 700, 'mois': 8, 'climat': 'chaud'
    }
    print('Profil JSON absent : profil de test utilisé')
print(profil)

Profil chargé : /kaggle/input/datasets/areejsallami/profil-utilisateur/profil_utilisateur (1).json
{'budget_total': 1500.0, 'jours': 5, 'mois': 7, 'climat': 'doux', 'interets': ['culture', 'adventure', 'nature'], 'destination_choisie': 'Geneva', 'pays_destination': 'Switzerland'}


## 7 — Recommander des activités par destination et préférences

Sélection **équilibrée** entre les intérêts demandés. Dans chaque catégorie, les fiches renseignées sont prioritaires ; aucune note ou popularité n'est inventée. Le budget et le mois ne filtrent pas les activités, car le dataset ne donne pas de prix chiffrés ni de saisonnalité fiable.

In [33]:
def recommander_activites(destination, preferences, top_n=15):
    if not destination or top_n < 1:
        return pd.DataFrame()
    preferences = [p for p in dict.fromkeys(preferences or [])
                   if p in categories_smarttrip]
    if not preferences:
        return pd.DataFrame()
    ville = str(destination).strip().lower()
    data_ville = activites_clean[
        (activites_clean['destination_norm'] == ville) &
        (activites_clean['preference_smarttrip'].isin(preferences))
    ].copy()
    if data_ville.empty:
        return pd.DataFrame()

    data_ville = data_ville.sort_values('completude', ascending=False, kind='stable')
    # Pour un même voyage, ne pas répéter un nom dans la même destination
    # (même si deux établissements distincts peuvent porter ce nom).
    data_ville = data_ville.drop_duplicates(
        subset=['country_norm','destination_norm','name_norm']
    )
    groupes = {
        p: data_ville[data_ville['preference_smarttrip'] == p].index.tolist()
        for p in preferences
    }
    selection = []
    while len(selection) < top_n and any(groupes.values()):
        for p in preferences:
            if groupes[p] and len(selection) < top_n:
                selection.append(groupes[p].pop(0))
    colonnes = ['country','destination','name','name_display','category','preference_smarttrip',
                'latitude','longitude','fee','opening_hours','website','osm_id']
    return data_ville.loc[selection, colonnes].reset_index(drop=True)

jours = int(profil['jours'])
activites_par_jour = 3
recommandations = recommander_activites(
    profil['destination_choisie'], profil['interets'],
    top_n=jours*activites_par_jour
)
print('Activités recommandées :', len(recommandations))
display(recommandations)

Activités recommandées : 0


""


In [34]:

# Séparer les activités recommandées
# selon le moment de la journée

def recommander_par_moment(destination, preferences,
                           moment, top_n=10):

    # Récupérer les préférences pertinentes
    prefs = list(preferences or [])

    if moment == "soir":
        # Ajouter gastronomie et vie nocturne
        for p in ["gastronomy", "nightlife"]:
            if p not in prefs:
                prefs.append(p)

    # Réutiliser le moteur de recommandation existant
    candidats = recommander_activites(
        destination, prefs, top_n=500
    )

    if candidats.empty:
        return candidats

    # Associer le moment à chaque activité
    candidats["moment"] = (
        candidats["category"]
        .astype(str).str.strip().str.lower()
        .apply(classifier_moment)
    )

    moment_cible = (
        "jour" if moment == "jour"
        else "soir_possible"
    )

    return (
        candidats[candidats["moment"] == moment_cible]
        .head(top_n)
        .reset_index(drop=True)
    )


destination = profil["destination_choisie"]
preferences = profil["interets"]

activites_jour = recommander_par_moment(
    destination, preferences, "jour", top_n=15
)

activites_soir = recommander_par_moment(
    destination, preferences, "soir", top_n=10
)

print("ACTIVITÉS DE JOUR")
display(activites_jour)

print("ACTIVITÉS DE SOIRÉE POSSIBLES")
display(activites_soir)


ACTIVITÉS DE JOUR


""


ACTIVITÉS DE SOIRÉE POSSIBLES


""


## 8 — Liste indicative sur la durée du voyage

Répartit les activités sans répétition sur les jours. **Ce n'est pas encore un itinéraire optimisé** : pas de temps de trajet, durée de visite ou garantie d'ouverture. Une optimisation géographique viendra ensuite.

In [35]:

import pandas as pd

# Nombre d'activités par jour
activites_jour_par_jour = 1
activites_soir_par_jour = 1

# Nombre de jours du voyage
jours = int(profil["jours"])

# Recommandations nécessaires
activites_jour = recommander_par_moment(
    profil["destination_choisie"],
    profil["interets"],
    "jour",
    top_n=jours * activites_jour_par_jour
)

activites_soir = recommander_par_moment(
    profil["destination_choisie"],
    profil["interets"],
    "soir",
    top_n=jours * activites_soir_par_jour
)

# Éviter les noms répétés
def supprimer_repetitions(df):
    # Aucune activité : retourner le tableau sans chercher "name".
    if df.empty:
        return df.copy()

    if "name" not in df.columns:
        raise ValueError(
            f"Colonne 'name' absente. Colonnes disponibles : "
            f"{df.columns.tolist()}"
        )

    df = df.copy()

    df["nom_normalise"] = (
        df["name"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    return (
        df.drop_duplicates(subset=["nom_normalise"])
        .drop(columns=["nom_normalise"])
    )

activites_jour = supprimer_repetitions(activites_jour)
activites_soir = supprimer_repetitions(activites_soir)

# Construire le planning
planning_lignes = []

for jour in range(1, jours + 1):

    # Activités de jour
    debut = (jour - 1) * activites_jour_par_jour
    fin = jour * activites_jour_par_jour

    for _, activite in activites_jour.iloc[debut:fin].iterrows():
        planning_lignes.append({
            "Jour": jour,
            "Moment": "Jour",
            "Activité": activite["name_display"],
            "Catégorie": activite["category"],
            "Préférence": activite["preference_smarttrip"],
            "latitude": activite["latitude"],
            "longitude": activite["longitude"],
            "opening_hours": activite["opening_hours"]
        })

    # Activité de soirée
    debut = (jour - 1) * activites_soir_par_jour
    fin = jour * activites_soir_par_jour

    for _, activite in activites_soir.iloc[debut:fin].iterrows():
        planning_lignes.append({
            "Jour": jour,
            "Moment": "Soirée possible",
            "Activité": activite["name_display"],
            "Catégorie": activite["category"],
            "Préférence": activite["preference_smarttrip"],
            "latitude": activite["latitude"],
            "longitude": activite["longitude"],
            "opening_hours": activite["opening_hours"]
        })

# Créer le DataFrame final
planning = pd.DataFrame(
    planning_lignes,
    columns=[
        "Jour", "Moment", "Activité",
        "Catégorie", "Préférence",
        "latitude", "longitude", "opening_hours"
    ]
)

display(planning)

# Vérifications
if not planning.empty:
    print(
        "Activités répétées :",
        planning["Activité"]
        .astype(str)
        .str.strip()
        .str.lower()
        .duplicated()
        .sum()
    )

    print(
        "Jours couverts :",
        planning["Jour"].nunique(),
        "/",
        jours
    )

    print(
        "Activités de jour :",
        (planning["Moment"] == "Jour").sum()
    )

    print(
        "Activités de soirée :",
        (planning["Moment"] == "Soirée possible").sum()
    )


,Jour,Moment,Activité,Catégorie,Préférence,latitude,longitude,opening_hours


## 9 — Exporter les données préparées

In [36]:

from pathlib import Path

# Dossier de sortie Kaggle
sortie = Path("/kaggle/working")
sortie.mkdir(parents=True, exist_ok=True)

# 1. Dataset nettoyé
activites_clean.to_csv(
    sortie / "all_activites_clean.csv",
    index=False
)

# 2. Doublons à vérifier
doublons_a_verifier.to_csv(
    sortie / "doublons_a_verifier.csv",
    index=False
)

# 3. Recommandations générales
recommandations.to_csv(
    sortie / "activites_recommandees.csv",
    index=False
)

# 4. Activités de jour
activites_jour.to_csv(
    sortie / "activites_jour.csv",
    index=False
)

# 5. Activités de soirée
activites_soir.to_csv(
    sortie / "activites_soir.csv",
    index=False
)

# 6. Planning complet jour / soirée
planning.to_csv(
    sortie / "planning_activites.csv",
    index=False
)

print("Exportation terminée !")
print("Fichiers créés dans :", sortie)

for fichier in sorted(sortie.glob("*.csv")):
    print("-", fichier.name)

print("Dataset original préservé.")


Exportation terminée !
Fichiers créés dans : /kaggle/working
- activites_jour.csv
- activites_recommandees.csv
- activites_soir.csv
- all_activites_clean.csv
- all_activites_clean_v2.csv
- doublons_a_verifier.csv
- planning_activites.csv
Dataset original préservé.


In [ ]:
#tester 

import requests
import os

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
UNSPLASH_KEY = user_secrets.get_secret("UNSPLASH_ACCESS_KEY")

def obtenir_photo_ville(ville, pays):
    url = "https://api.unsplash.com/search/photos"

    params = {
        "query": f"{ville} {pays} city travel",
        "per_page": 1,
        "orientation": "landscape"
    }

    headers = {
        "Authorization": f"Client-ID {UNSPLASH_KEY}"
    }

    response = requests.get(
        url, params=params, headers=headers, timeout=15
    )
    response.raise_for_status()

    photos = response.json().get("results", [])

    if not photos:
        return None

    photo = photos[0]

    return {
        "image_url": photo["urls"]["regular"],
        "photographer": photo["user"]["name"],
        "photographer_url": photo["user"]["links"]["html"],
        "unsplash_url": photo["links"]["html"]
    }

photo = obtenir_photo_ville("Rome", "Italy")
print(photo)


In [ ]:
from IPython.display import Image, display

display(Image(url=photo["image_url"], width=650))

In [ ]:
from IPython.display import display, HTML
from html import escape

destinations = [
    ("Paris", "France"),
    ("Rome", "Italy"),
    ("Tokyo", "Japan")
]

for ville, pays in destinations:
    photo = obtenir_photo_ville(ville, pays)

    if photo:
        display(HTML(f"<h3>{escape(ville)}, {escape(pays)}</h3>"))
        display(Image(url=photo["image_url"], width=450))